In [1]:
from dotenv import load_dotenv

In [3]:
from langchain_teddynote import logging
import os
import warnings

logging.langsmith("CH08-Embeddings")

warnings.filterwarnings("ignore")

os.environ["HF_HOME"] = "./cache/"

LangSmith 추적을 시작합니다.
[프로젝트명]
CH08-Embeddings


In [8]:
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.",
    "LangCahin은 초거대 언어모델로  애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.",
]

In [9]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embeddings = HuggingFaceEndpointEmbeddings(
    model=model_name,
    task="feature-extraction",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"]
)

In [10]:
%%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 234 ms
Wall time: 5.58 s


In [11]:
print("[HuggingFace Endpoint Embedding]")
print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

[HuggingFace Endpoint Embedding]
Model: 		intfloat/multilingual-e5-large-instruct
Dimension: 	1024


In [12]:
embedded_query = hf_embeddings.embed_query("LangChain에 대해서 알려주세요.")
embedded_query

[0.007985987700521946,
 0.01688126102089882,
 0.0009302098187617958,
 -0.024559255689382553,
 0.027032550424337387,
 -0.017867881804704666,
 -0.022472545504570007,
 0.03082295134663582,
 0.031417835503816605,
 -0.035394102334976196,
 0.02045944705605507,
 0.015117725357413292,
 -0.025928055867552757,
 -0.00048280751798301935,
 -0.01964387483894825,
 -0.019827160984277725,
 -0.06975308060646057,
 0.007183140143752098,
 -0.028460603207349777,
 -0.010040021501481533,
 0.054408423602581024,
 0.01219356618821621,
 -0.010757098905742168,
 -0.02073003724217415,
 -0.013305061496794224,
 -0.010071979835629463,
 -0.030244380235671997,
 -0.044592518359422684,
 0.0011336494935676455,
 -0.036801137030124664,
 0.0030633267015218735,
 0.007607906591147184,
 -0.01647542603313923,
 -0.044309940189123154,
 -0.01846487447619438,
 0.02911071851849556,
 0.05282043665647507,
 0.04078085720539093,
 -0.02183525077998638,
 0.05975455418229103,
 -0.023669712245464325,
 0.06193467602133751,
 0.023379752412438393

In [13]:
import numpy as np
np.array(embedded_query) @np.array(embedded_documents).T

array([0.84281223, 0.86560872, 0.86114495, 0.88275677, 0.77247184])

In [17]:
stored_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]
stored_idx

array([3, 1, 2, 0, 4])

In [19]:
print("[Query] LangChain 에 대해서 알려주세요. \ㅜ ================================")
for i, idx in enumerate(stored_idx):
    print(f"[{i}] {texts[idx]}")
    print()

[Query] LangChain 에 대해서 알려주세요. \ㅜ ================================
[0] LangCahin은 초거대 언어모델로  애플리케이션을 구축하는 과정을 단순화합니다.

[1] LangChain simplifies the process of building applications with large language models

[2] 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.

[3] 안녕, 만나서 반가워.

[4] Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.



In [23]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    # model_kwargs={"device": "mps"},  # cuda, cpu
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 898.91it/s]


In [24]:
%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 0 ns
Wall time: 0 ns


In [25]:
print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

Model: 		intfloat/multilingual-e5-large-instruct
Dimension: 	1024


In [26]:
#BGE-M3
model_name = "BAAI/bge-m3"
model_kwargs = {"device":"cpu"}
encode_kwargs = {"normalize_embeddings":True}
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

%time
embedded_documents = hf_embeddings.embed_documents(texts)

print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 43847.20it/s]


CPU times: total: 0 ns
Wall time: 0 ns
Model: 		BAAI/bge-m3
Dimension: 	1024


In [ ]:
import numpy as np

embedded_query = hf_embeddings.embed_query("LangChain에 대해서 알려주세요.")
embedded_documents = hf_embeddings.embed_documents(texts)

np.array(embedded_query) @ np.array(embedded_documents).T

sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()

In [31]:
print("[Query] LangCahin에 대해서 알려주세요. \n=================================")
for i, idx in enumerate(sorted_idx):
    print(f"[{i}] {texts[idx]}")
    print()

[Query] LangCahin에 대해서 알려주세요. 
[0] LangChain simplifies the process of building applications with large language models

[1] 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.

[2] LangCahin은 초거대 언어모델로  애플리케이션을 구축하는 과정을 단순화합니다.

[3] 안녕, 만나서 반가워.

[4] Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.



In [32]:
from FlagEmbedding import BGEM3FlagModel

model_name = "BAAI/bge-m3"
bge_embeddings = BGEM3FlagModel(
    model_name, use_fp16=True
)

bge_embedded = bge_embeddings.encode(
    texts,
    batch_size=12,
    max_length=8192,
)["dense_vecs"]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 898.76it/s]


In [33]:
bge_embedded.shape

(5, 1024)

In [36]:
from FlagEmbedding import BGEM3FlagModel

bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=True
)

bge_encoded = bge_flagmodel.encode(texts, return_dense=True)
bge_encoded["dense_vecs"].shape

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1369.43it/s]


(5, 1024)

In [37]:
bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=True
)
bge_encoded = bge_flagmodel.encode(texts, return_sparse=True)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 780.83it/s]


In [38]:
lexical_scores1 = bge_flagmodel.compute_lexical_matching_score(
    bge_encoded["lexical_weights"][0], bge_encoded["lexical_weights"][0]
)
lexical_scores2 = bge_flagmodel.compute_lexical_matching_score(
    bge_encoded["lexical_weights"][0], bge_encoded["lexical_weights"][1]
)

print(lexical_scores1)
print(lexical_scores2)

0.30156046
0


In [39]:
bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=True
)
bge_encoded = bge_flagmodel.encode(texts, return_colbert_vecs=True)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1694.24it/s]


In [40]:
colbert_scores1 = bge_flagmodel.colbert_score(
    bge_encoded["colbert_vecs"][0], bge_encoded["colbert_vecs"][0]
)
colbert_scores2 = bge_flagmodel.colbert_score(
    bge_encoded["colbert_vecs"][0], bge_encoded["colbert_vecs"][1]
)

print(colbert_scores1)
print(colbert_scores2)

tensor(1.)
tensor(0.3748)
